# 04 — Strategies

**Workstream D** of the group project plan. This notebook builds the eight weight functions, each with a
stated objective, constraints and parameters. It tests them, and it runs the two checks the brief
asks for explicitly: whether **IV differs from ERC**, and whether **MV is distinct from GMV and MSR**.
It produces **no performance results**. Those come from notebook 05.

| Task | Output |
| --- | --- |
| D1–D8 | `strategies.STRATEGIES`: EW, GMV, MV, MSR, IV, ERC, MDP, MDC |
| D9 | One signature, `f(returns) -> weights`, interchangeable in `backtest.run_backtest`, with unit tests |
| D10 | **Table 3.2 — Strategy specifications**, including the shortcomings column the group's draft asks for |
| D11 | **Table 3.3 — IV against ERC** on one fixed window, plus the same comparison across all 400 rebalances |
| D12 | **Table 3.4 — MV, GMV and MSR** on the same window, plus a distinctness count across all 400 |

Shared code: `src/strategies.py`. The engine is `src/backtest.py` (notebook 03).

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.optimize import minimize

from src import backtest as bt
from src import config as cfg
from src import data, fmt
from src import strategies as st

np.random.seed(cfg.SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

---

## Data and the 400 estimation windows

The panel and the 100 experiments are rebuilt exactly as in notebook 03. Each experiment rebalances
four times, so a strategy is asked for weights on **400 estimation windows**. All of them are built
here with the engine's own slicer, so the diagnostics below see exactly the data the backtest will.

In [2]:
close, volume, extraction_date = data.download_raw()
panel = data.build_panel(close, volume, extraction_date)
prices = panel.prices[cfg.ASSET_ORDER]
schedule = bt.Schedule()
experiments = bt.sample_experiments(prices, schedule)

windows = []   # (experiment, rebalance, returns window, daily risk-free over the window)
for i, e in experiments.iterrows():
    returns = bt.experiment_prices(prices, e).pct_change().iloc[1:]
    for k, execution in enumerate(schedule.execution_positions()):
        formation = execution - schedule.lag
        window = bt.estimation_window(returns, formation,
                                      schedule.initial_lookback if k == 0 else schedule.lookback)
        rf = panel.risk_free.reindex(window.index).ffill() / 100 / cfg.TRADING_DAYS
        windows.append((i, k, window, rf))

print(f"Extraction date : {panel.extraction_date.isoformat()}")
print(f"Windows         : {len(windows)} ({len(experiments)} experiments x "
      f"{len(schedule.execution_positions())} rebalances)")

Extraction date : 2026-10-07
Windows         : 400 (100 experiments x 4 rebalances)


---

## Estimation inputs, shared by all eight

| Input | Estimator | Used by |
| --- | --- | --- |
| μ | Sample mean of daily simple returns × 252 | MV, MSR |
| Σ | **Sample covariance** × 252 | GMV, MV, MSR, ERC, MDP |
| σ | √diag(Σ) | IV, MDP |
| C | Correlation matrix from Σ | MDC |
| r_f | Mean `^IRX` over the window, annualised | MSR |

**Why the sample covariance, not Ledoit–Wolf.** Shrinkage exists for when there are many assets and few
observations. Here there are 4 assets and 252–503 observations, so the sample matrix is well
conditioned. Shrinkage also has a cost specific to this brief. The Ledoit–Wolf constant-correlation
target pulls every correlation towards the average, and **when all correlations are equal, ERC and MDP
reduce exactly to IV** (test 6 below). Shrinkage would therefore erase the very contrast the brief
asks us to show. The shrinkage intensity it would choose on these windows:

In [3]:
intensity = pd.Series([st.estimate(w, covariance="ledoit-wolf").shrinkage for _, _, w, _ in windows])
print("Ledoit-Wolf intensity across the 400 windows (0 = sample matrix, 1 = constant correlation):")
print(intensity.describe(percentiles=[0.1, 0.5, 0.9]).round(2).to_string())

Ledoit-Wolf intensity across the 400 windows (0 = sample matrix, 1 = constant correlation):
count    400.00
mean       0.56
std        0.34
min        0.04
10%        0.12
50%        0.52
90%        1.00
max        1.00


The median window would be pushed about halfway to constant correlation, and many all the way. The
daily returns are fat-tailed (notebook 02), which inflates the estimated noise in the sample matrix.
Ledoit–Wolf stays in `src/strategies.py` as a switch (`COVARIANCE = "ledoit-wolf"`) for a robustness
re-run in workstream F. It is not the baseline.

---

## D10 — Table 3.2, strategy specifications

Every strategy is long-only, fully invested and unlevered, the common baseline of definitions §7. They
differ in objective and in which estimates they consume. The *shortcomings* column comes from the
group's draft (W-4 in the work-division page). It feeds the trade-off discussion in report section 5.

In [4]:
spec = st.specification_table()
print("Table 3.2 - Strategy specifications. All eight are long-only, fully invested and unlevered; "
      "μ and Σ annualised (x 252) from daily simple returns over the estimation window.")
display(spec)

Table 3.2 - Strategy specifications. All eight are long-only, fully invested and unlevered; μ and Σ annualised (x 252) from daily simple returns over the estimation window.


,Objective,Constraints,Parameters,Estimation inputs,Shortcomings
Strategy,,,,,
EW,wᵢ = 1/N,"w ≥ 0, Σw = 1",—,None,Ignores risk and correlation entirely; the risk is dominated by the most volatile asset
GMV,min wᵀΣw,"w ≥ 0, Σw = 1",—,Σ,Ignores expected returns; tends to concentrate in the lowest-volatility asset
MV,max wᵀμ − (λ/2)wᵀΣw,"w ≥ 0, Σw = 1",λ = 3 (annual units),"μ, Σ",Very sensitive to errors in μ; often ends up in a corner holding one or two assets
MSR,max (wᵀμ − r_f) / √(wᵀΣw),"w ≥ 0, Σw = 1",r_f = mean ^IRX over the window,"μ, Σ, r_f","As MV, plus undefined when no asset beats r_f (logged as a failure)"
IV,wᵢ ∝ 1/σᵢ,"w ≥ 0, Σw = 1",—,σ (diagonal of Σ),"Ignores correlations, so two highly correlated assets get full weight each"
ERC,wᵢ(Σw)ᵢ = wᵀΣw / N for all i,"w ≥ 0, Σw = 1",Equal risk budgets 1/N,Σ,Low-risk assets get large weights; the result depends on the quality of the Σ estimate
MDP,max (wᵀσ) / √(wᵀΣw),"w ≥ 0, Σw = 1",—,"σ, Σ",Favours low-correlation assets whatever their return; can concentrate
MDC,"min wᵀCw, C = correlation matrix","w ≥ 0, Σw = 1",—,C,"Treats every asset as equally volatile, so it can overweight a very risky asset that is uncorrelated with the rest"


**How the optimisers are solved.** GMV, MV, MSR, MDP and MDC are all one problem,
*min ½xᵀQx + cᵀx subject to aᵀx = b, x ≥ 0*, with different Q, c and a:

| Strategy | Q | c | a |
| --- | --- | --- | --- |
| GMV | Σ | 0 | 1 |
| MV | λΣ | −μ | 1 |
| MSR | Σ | 0 | μ − r_f, then w = x / Σx |
| MDP | Σ | 0 | σ, then w = x / Σx |
| MDC | C | 0 | 1 |

MSR and MDP maximise a ratio, which is not directly a QP. The substitution *y = w / (aᵀw)* turns each
into the QP above (Cornuéjols and Tütüncü's standard trick). `solve_qp` solves it **exactly**: it
enumerates the possible sets of non-zero weights, at most 15 for four assets, and solves the optimality
(KKT) conditions on each. There is no iterative solver, so no tolerance to tune and no convergence
failure. The only failure left is a problem with no solution. ERC has no QP form. It is solved by
Newton's method on Spinu's (2013) convex reformulation, and the risk contributions are then checked to
be equal to 1e-8.

---

## D9 — Unit tests

Each test checks a strategy against a case with a known answer, or checks a property that must hold on
every one of the 400 real windows. The `returns_with` helper builds a synthetic return panel whose
sample mean and covariance are **exactly** a chosen μ and Σ. That lets a test state the answer before
calling the strategy.

In [5]:
def returns_with(mu, cov, t=252, seed=0):
    # Daily returns whose sample mean x 252 is exactly `mu` and whose sample covariance x 252 is exactly `cov`.
    mu, cov = np.asarray(mu, float), np.asarray(cov, float)
    n = len(mu)
    z = np.random.default_rng(seed).normal(size=(t, n))
    z -= z.mean(axis=0)
    z = z @ np.linalg.inv(np.linalg.cholesky(np.cov(z, rowvar=False))).T        # sample cov = I
    x = z @ np.linalg.cholesky(cov / cfg.TRADING_DAYS).T + mu / cfg.TRADING_DAYS
    return pd.DataFrame(x, columns=[f"A{i}" for i in range(n)])


def constant_correlation(vol, rho):
    vol = np.asarray(vol, float)
    corr = np.full((len(vol), len(vol)), rho)
    np.fill_diagonal(corr, 1.0)
    return corr * np.outer(vol, vol)


VOL = [0.10, 0.20, 0.30, 0.60]
EXACT = 1e-9


def close_to(w, expected, tol=EXACT):
    np.testing.assert_allclose(np.asarray(w, float), np.asarray(expected, float) / np.sum(expected), atol=tol)


# ---- 1. The exact QP solver agrees with a general-purpose optimiser -------- #
def test_1_qp_matches_slsqp():
    rng = np.random.default_rng(1)
    for _ in range(300):
        n = int(rng.integers(2, 7))
        a_ = rng.normal(size=(n, n))
        q, c = a_ @ a_.T + 0.01 * np.eye(n), rng.normal(size=n)
        x = st.solve_qp(q, c, np.ones(n))
        ref = minimize(lambda z: 0.5 * z @ q @ z + c @ z, np.full(n, 1 / n), jac=lambda z: q @ z + c,
                       bounds=[(0, None)] * n, method="SLSQP",
                       constraints=[{"type": "eq", "fun": lambda z: z.sum() - 1}],
                       options={"ftol": 1e-14, "maxiter": 500})
        assert x.min() >= 0 and np.isclose(x.sum(), 1)
        # Never worse than SLSQP, to rounding.
        assert 0.5 * x @ q @ x + c @ x <= ref.fun + 1e-12


# ---- 2. The synthetic panel really has the stated moments ------------------ #
def test_2_returns_with_is_exact():
    cov = constant_correlation(VOL, 0.3)
    est = st.estimate(returns_with([0.05, 0.06, 0.07, 0.08], cov))
    np.testing.assert_allclose(est.cov.to_numpy(), cov, atol=1e-12)
    np.testing.assert_allclose(est.mu.to_numpy(), [0.05, 0.06, 0.07, 0.08], atol=1e-12)


# ---- 3. EW and GMV against closed forms ------------------------------------ #
def test_3_ew_and_gmv():
    r = returns_with(np.zeros(4), np.diag(np.square(VOL)))
    close_to(st.equal_weight(r), np.ones(4))
    close_to(st.global_minimum_variance(r), 1 / np.square(VOL))                 # uncorrelated: w ∝ 1/σ²
    # Correlated, interior solution: w ∝ Σ⁻¹1.
    cov = constant_correlation(VOL[:3], 0.2)
    close_to(st.global_minimum_variance(returns_with(np.zeros(3), cov)), np.linalg.solve(cov, np.ones(3)))


# ---- 4. MV spans the frontier: λ -> ∞ is GMV, λ -> 0 is the top-μ asset ---- #
def test_4_mv_limits():
    r = returns_with([0.02, 0.05, 0.09, 0.15], constant_correlation(VOL, 0.2))
    close_to(st.mean_variance(r, risk_aversion=1e7), st.global_minimum_variance(r), tol=1e-6)
    close_to(st.mean_variance(r, risk_aversion=1e-7), [0, 0, 0, 1])
    # λ = 3 is neither end.
    w = st.mean_variance(r)
    assert (w - st.global_minimum_variance(r)).abs().max() > 0.05 and w.max() < 1


# ---- 5. MSR: tangency formula when interior, raises when undefined --------- #
def test_5_msr():
    mu = np.array([0.06, 0.08, 0.10, 0.14])
    cov = constant_correlation(VOL, 0.2)
    rf = pd.Series(0.02 / cfg.TRADING_DAYS, index=range(252))
    tangency = np.linalg.solve(cov, mu - 0.02)
    assert (tangency > 0).all()                                                # interior case
    close_to(st.maximum_sharpe(returns_with(mu, cov), risk_free=rf), tangency)
    try:
        st.maximum_sharpe(returns_with(np.full(4, 0.01), cov), risk_free=rf)
    except st.InfeasibleError:
        pass
    else:
        raise AssertionError("MSR returned weights when no asset beats the risk-free rate")
    try:
        st.maximum_sharpe(returns_with(mu, cov))
    except ValueError:
        pass
    else:
        raise AssertionError("MSR ran without a risk-free rate")


# ---- 6. IV, ERC and MDP: identical exactly when correlations are equal ----- #
def test_6_iv_erc_mdp_under_constant_correlation():
    for rho in (0.0, 0.3, 0.7):
        r = returns_with(np.zeros(4), constant_correlation(VOL, rho))
        iv = st.inverse_volatility(r)
        close_to(iv, 1 / np.array(VOL))
        close_to(st.equal_risk_contribution(r), iv.to_numpy())
        close_to(st.most_diversified(r), iv.to_numpy())
    # ...and different once correlations differ: assets 0 and 1 highly correlated, 2 and 3 not.
    corr = np.array([[1, .8, 0, 0], [.8, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
    r = returns_with(np.zeros(4), corr * np.outer(VOL, VOL))
    erc, iv = st.equal_risk_contribution(r), st.inverse_volatility(r)
    assert (erc - iv).iloc[:2].lt(0).all() and (erc - iv).iloc[2:].gt(0).all()   # ERC cuts the correlated pair


# ---- 7. MDC: equal weight when uncorrelated, ignores volatility ------------ #
def test_7_mdc():
    close_to(st.maximum_decorrelation(returns_with(np.zeros(4), np.diag(np.square(VOL)))), np.ones(4))
    corr = constant_correlation([1, 1, 1], 0.4)
    a = st.maximum_decorrelation(returns_with(np.zeros(3), corr * np.outer([.1, .2, .3], [.1, .2, .3])))
    b = st.maximum_decorrelation(returns_with(np.zeros(3), corr * np.outer([.5, .1, .9], [.5, .1, .9])))
    close_to(a, b.to_numpy())                                                  # same C, same weights


# ---- 8. ERC equalises risk on every real window ---------------------------- #
def test_8_erc_on_real_windows():
    for _, _, w, _ in windows:
        rc = st.risk_contributions(st.equal_risk_contribution(w), st.estimate(w).cov)
        assert (rc - 0.25).abs().max() < 1e-8


# ---- 9. All eight give valid weights on every real window ------------------ #
def test_9_valid_on_all_windows():
    for name, fn in st.STRATEGIES.items():
        for _, _, w, rf in windows:
            try:
                weights = fn(w, risk_free=rf) if name == "MSR" else fn(w)
            except st.InfeasibleError:
                assert name == "MSR", name                                     # only MSR may be undefined
                continue
            assert list(weights.index) == list(w.columns)
            assert weights.min() >= 0 and abs(weights.sum() - 1) < 1e-12, name


# ---- 10. Ledoit-Wolf, kept as a switch, behaves as specified ----------------- #
def test_10_ledoit_wolf():
    for _, _, w, _ in windows[:40]:
        est, sample = st.estimate(w, covariance="ledoit-wolf"), st.estimate(w)
        assert 0 <= est.shrinkage <= 1
        np.testing.assert_allclose(np.diag(est.cov), np.diag(sample.cov), rtol=1e-12)   # variances kept
        assert np.linalg.eigvalsh(est.cov.to_numpy()).min() > 0


TESTS = [test_1_qp_matches_slsqp, test_2_returns_with_is_exact, test_3_ew_and_gmv, test_4_mv_limits,
         test_5_msr, test_6_iv_erc_mdp_under_constant_correlation, test_7_mdc,
         test_8_erc_on_real_windows, test_9_valid_on_all_windows, test_10_ledoit_wolf]

In [6]:
outcomes = []
for test in TESTS:
    try:
        test()
        outcomes.append((test.__name__, "PASS", ""))
    except Exception as exc:
        outcomes.append((test.__name__, "FAIL", f"{type(exc).__name__}: {exc}"))

test_report = pd.DataFrame(outcomes, columns=["Test", "Result", "Detail"]).set_index("Test")
display(test_report)
assert (test_report["Result"] == "PASS").all(), "unit tests failed"
print(f"All {len(TESTS)} tests pass.")

,Result,Detail
Test,,
test_1_qp_matches_slsqp,PASS,
test_2_returns_with_is_exact,PASS,
test_3_ew_and_gmv,PASS,
test_4_mv_limits,PASS,
test_5_msr,PASS,
test_6_iv_erc_mdp_under_constant_correlation,PASS,
test_7_mdc,PASS,
test_8_erc_on_real_windows,PASS,
test_9_valid_on_all_windows,PASS,


All 10 tests pass.


---

## D11 — Does IV differ from ERC?

IV sizes each asset by its own volatility alone. ERC sizes it by its contribution to *portfolio* risk,
which also depends on how it co-moves with the others. The two coincide exactly when every pair has the
same correlation (test 6). They separate as correlations become unequal: ERC gives **less** weight than
IV to an asset that is highly correlated with the rest, and **more** to one that diversifies.

**One fixed window.** Experiment 1's opening estimate, taken as the first window drawn rather than
picked for effect:

In [7]:
FIXED = 0                                         # experiment 1, rebalance 0
exp_id, _, fixed, fixed_rf = windows[FIXED]
est = st.estimate(fixed)
iv, erc = st.inverse_volatility(fixed), st.equal_risk_contribution(fixed)

corr = est.corr
avg_corr = (corr.sum() - 1) / (len(corr) - 1)
table_3_3 = pd.DataFrame({
    "Volatility %": est.vol * 100,
    "Avg. correlation with others": avg_corr,
    "IV weight %": iv * 100,
    "ERC weight %": erc * 100,
    "ERC − IV, pp": (erc - iv) * 100,
    "IV risk share %": st.risk_contributions(iv, est.cov) * 100,
    "ERC risk share %": st.risk_contributions(erc, est.cov) * 100,
})
table_3_3.index.name = "Ticker"

print(f"Table 3.3 - IV against ERC on one estimation window: experiment {exp_id}, "
      f"{fixed.index[0].date()} to {fixed.index[-1].date()}, n = {len(fixed)} daily returns. "
      "Risk share = each asset's share of portfolio variance.")
display(fmt.style(table_3_3, {
    "Volatility %": lambda v: fmt.percent(v, 1),
    "Avg. correlation with others": lambda v: fmt.number(v, 2),
    "IV weight %": lambda v: fmt.percent(v, 1),
    "ERC weight %": lambda v: fmt.percent(v, 1),
    "ERC − IV, pp": lambda v: fmt.number(v, 1),
    "IV risk share %": lambda v: fmt.percent(v, 1),
    "ERC risk share %": lambda v: fmt.percent(v, 1),
}))
print(corr.round(2).to_string())

Table 3.3 - IV against ERC on one estimation window: experiment 1, 2020-07-06 to 2022-07-01, n = 503 daily returns. Risk share = each asset's share of portfolio variance.


,Volatility %,Avg. correlation with others,IV weight %,ERC weight %,"ERC − IV, pp",IV risk share %,ERC risk share %
Ticker,,,,,,,
GC=F,16.1,0.12,21.4,20.0,-1.5,26.4,25.0
GOVT,5.4,0.05,64.6,66.6,2.0,22.3,25.0
TSM,37.4,0.10,9.2,9.0,-0.3,25.2,25.0
BTC-USD,72.3,0.12,4.8,4.5,-0.3,26.1,25.0


Ticker   GC=F  GOVT   TSM  BTC-USD
Ticker                            
GC=F     1.00  0.22  0.09     0.05
GOVT     0.22  1.00 -0.08     0.01
TSM      0.09 -0.08  1.00     0.29
BTC-USD  0.05  0.01  0.29     1.00


**Across all 400 windows.** If the difference tracks the correlation structure, an asset's ERC − IV
weight gap should fall as its average correlation with the other three rises. The gap and the
correlation are both measured relative to the window's own average, so that windows with different
correlation levels are comparable:

In [8]:
rows = []
for i, k, w, _ in windows:
    est_w = st.estimate(w)
    c = est_w.corr
    avg = (c.sum() - 1) / (len(c) - 1)
    gap = st.equal_risk_contribution(w) - st.inverse_volatility(w)
    for asset in w.columns:
        rows.append({"experiment": i, "rebalance": k, "asset": asset, "gap_pp": gap[asset] * 100,
                     "avg_corr_rel": avg[asset] - avg.mean(),
                     "corr_dispersion": c.where(~np.eye(len(c), dtype=bool)).stack().std()})
gaps = pd.DataFrame(rows)
per_window = gaps.groupby(["experiment", "rebalance"]).agg(max_gap=("gap_pp", lambda g: g.abs().max()),
                                                           dispersion=("corr_dispersion", "first"))

slope_corr = gaps["gap_pp"].corr(gaps["avg_corr_rel"])
size_corr = per_window["max_gap"].corr(per_window["dispersion"])
iv_erc_summary = pd.DataFrame({
    "Value": [per_window["max_gap"].median(), per_window["max_gap"].quantile(0.9), per_window["max_gap"].max(),
              slope_corr, size_corr],
}, index=["Largest |ERC − IV| per window, median (pp)", "... 90th percentile (pp)", "... maximum (pp)",
          "Corr(ERC − IV gap, asset's relative avg. correlation)",
          "Corr(largest gap, dispersion of pairwise correlations)"])
display(fmt.style(iv_erc_summary, {"Value": lambda v: fmt.number(v, 2)}))
assert slope_corr < -0.5 and size_corr > 0.3

,Value
"Largest |ERC − IV| per window, median (pp)",2.58
... 90th percentile (pp),5.77
... maximum (pp),9.80
"Corr(ERC − IV gap, asset's relative avg. correlation)",-0.91
"Corr(largest gap, dispersion of pairwise correlations)",0.75


**Reading.** IV and ERC differ in every window, and in the direction theory predicts. The more
correlated an asset is with the rest, the more ERC cuts it relative to IV (a strongly negative
correlation in the first diagnostic). The windows where correlations are most unequal produce the
largest gaps (a positive correlation in the second). The **size** of the difference is modest, a few
percentage points in a typical window. That is a property of this universe, not of the method: the
six assets are weakly and fairly evenly correlated (Figure 1.1), the case in which ERC stays close to
IV. The report should state both points. The difference is real and correlation-driven, and it is
small here because the correlations are low.

---

## D12 — Are MV, GMV and MSR three different portfolios?

All three sit on the same long-only efficient frontier. GMV is its minimum-risk end. MSR is the point
with the highest Sharpe ratio. MV is the point the risk-aversion coefficient λ = 3 selects. The pre-set
λ is what makes MV a separate portfolio. On the fixed window:

In [9]:
gmv, mv, msr = (st.global_minimum_variance(fixed), st.mean_variance(fixed),
                st.maximum_sharpe(fixed, risk_free=fixed_rf))
table_3_4 = pd.DataFrame({"μ %/yr": est.mu * 100, "σ %/yr": est.vol * 100,
                          "GMV %": gmv * 100, "MV (λ = 3) %": mv * 100, "MSR %": msr * 100})
table_3_4.index.name = "Ticker"

def portfolio_stats(w):
    return pd.Series({"μ %/yr": w @ est.mu * 100, "σ %/yr": np.sqrt(w @ est.cov @ w) * 100})

for name, w in (("GMV", gmv), ("MV (λ = 3)", mv), ("MSR", msr)):
    s = portfolio_stats(w)
    print(f"{name:<11} ex-ante μ {s['μ %/yr']:6.1f}%  σ {s['σ %/yr']:5.1f}%")

rf_annual = fixed_rf.mean() * cfg.TRADING_DAYS * 100
print(f"\nTable 3.4 - MV, GMV and MSR weights on one estimation window: experiment {exp_id}, "
      f"{fixed.index[0].date()} to {fixed.index[-1].date()}, n = {len(fixed)}; r_f = {rf_annual:.2f}%/yr. "
      "μ and σ are in-sample estimates, not results.")
display(fmt.style(table_3_4, {c: lambda v: fmt.percent(v, 1) for c in table_3_4.columns}))

GMV         ex-ante μ   -5.0%  σ   5.2%
MV (λ = 3)  ex-ante μ   31.5%  σ  33.3%
MSR         ex-ante μ   41.9%  σ  44.0%

Table 3.4 - MV, GMV and MSR weights on one estimation window: experiment 1, 2020-07-06 to 2022-07-01, n = 503; r_f = 0.21%/yr. μ and σ are in-sample estimates, not results.


,μ %/yr,σ %/yr,GMV %,MV (λ = 3) %,MSR %
Ticker,,,,,
GC=F,1.6,16.1,2.8,28.0,4.9
GOVT,-6.0,5.4,94.3,0.0,0.0
TSM,22.4,37.4,2.9,36.0,45.5
BTC-USD,63.7,72.3,0.0,36.0,49.6


**Across all 400 windows.** The distance between two weight vectors is measured as **½ Σ|wᵢ − vᵢ|**:
the share of the portfolio that would have to change hands to turn one into the other. Two portfolios
**coincide** when that is below 0.01%.

In [10]:
def distance(a, b):
    return 0.5 * (a - b).abs().sum()

rows, msr_weights = [], []
for i, k, w, rf in windows:
    g, m = st.global_minimum_variance(w), st.mean_variance(w)
    try:
        s = st.maximum_sharpe(w, risk_free=rf)
    except st.InfeasibleError:
        s = None
    msr_weights.append(s)
    rows.append({"experiment": i, "rebalance": k,
                 "GMV–MV": distance(g, m),
                 "GMV–MSR": np.nan if s is None else distance(g, s),
                 "MV–MSR": np.nan if s is None else distance(m, s),
                 "mv_single": m.max() > 1 - 1e-9,
                 "msr_single": False if s is None else s.max() > 1 - 1e-9,
                 "mv_asset": m.idxmax()})
d12 = pd.DataFrame(rows)

COINCIDE = 1e-4
pairs = ["GMV–MV", "GMV–MSR", "MV–MSR"]
table_d12 = pd.DataFrame({
    "Windows compared": [d12[p].notna().sum() for p in pairs],
    "Median distance %": [d12[p].median() * 100 for p in pairs],
    "10th pct. distance %": [d12[p].quantile(0.1) * 100 for p in pairs],
    "Coincide": [(d12[p] < COINCIDE).sum() for p in pairs],
}, index=pd.Index(pairs, name="Pair"))
print("Distinctness of GMV, MV (λ = 3) and MSR across the 400 estimation windows. "
      f"MSR undefined (no asset above r_f) in {d12['GMV–MSR'].isna().sum()} windows.")
display(fmt.style(table_d12, {"Windows compared": fmt.count, "Coincide": fmt.count,
                              "Median distance %": lambda v: fmt.percent(v, 1),
                              "10th pct. distance %": lambda v: fmt.percent(v, 1)}))

same = d12[d12["MV–MSR"] < COINCIDE]
print(f"MV = MSR in {len(same)} windows; in {int((same['mv_single'] & same['msr_single']).sum())} of them "
      "both hold 100% of one asset.")
print("That asset:", same["mv_asset"].value_counts().to_dict())
print(f"MV holds 100% of one asset in {d12['mv_single'].mean() * 100:.1f}% of all windows.")

Distinctness of GMV, MV (λ = 3) and MSR across the 400 estimation windows. MSR undefined (no asset above r_f) in 7 windows.


,Windows compared,Median distance %,10th pct. distance %,Coincide
Pair,,,,
GMV–MV,400,95.2,63.5,0
GMV–MSR,393,69.6,12.0,0
MV–MSR,393,44.5,2.2,30


MV = MSR in 30 windows; in 30 of them both hold 100% of one asset.
That asset: {'RNMBY': 13, 'GC=F': 5, 'TSM': 4, 'VNQ': 3, 'GOVT': 3, 'BTC-USD': 2}
MV holds 100% of one asset in 29.5% of all windows.


**Context for the λ decision: the same count at other values of λ.** This is a weight diagnostic only.
No performance figure has been computed for any λ.

In [11]:
lam_rows = []
for lam in (1, 2, 3, 5, 10, 20):
    mv_l = [st.mean_variance(w, risk_aversion=lam) for _, _, w, _ in windows]
    coincide = sum(s is not None and distance(m_, s) < COINCIDE for m_, s in zip(mv_l, msr_weights))
    lam_rows.append({"λ": lam, "MV = MSR (windows)": coincide,
                     "MV all in one asset %": np.mean([m_.max() > 1 - 1e-9 for m_ in mv_l]) * 100,
                     "Median distance to GMV %": np.median([distance(m_, st.global_minimum_variance(w))
                                                            for m_, (_, _, w, _) in zip(mv_l, windows)]) * 100})
lam_table = pd.DataFrame(lam_rows).set_index("λ")
display(fmt.style(lam_table, {"MV = MSR (windows)": fmt.count,
                              "MV all in one asset %": lambda v: fmt.percent(v, 1),
                              "Median distance to GMV %": lambda v: fmt.percent(v, 1)}))

,MV = MSR (windows),MV all in one asset %,Median distance to GMV %
λ,,,
1,52,74.2,97.8
2,49,56.0,96.8
3,30,29.5,95.2
5,11,7.8,90.4
10,5,2.0,53.6
20,2,0.5,27.4


**Reading, and the λ decision.** MV is never close to GMV, and GMV never coincides with MSR. **MV and
MSR coincide in a minority of windows, and every one of those is a corner.** Both put 100% in the same
asset, usually one with an extreme trailing return. On a corner, the risk penalty in MV is too weak to
pull the portfolio off the top-return asset, and the same asset also has the best Sharpe ratio. Higher
λ would remove most of these coincidences (table above).

The group **kept λ = 3**, the value fixed before any run (definitions §7). That reflects a moderately
risk-averse investor in annual units, and the coincidences are reported rather than tuned away. Report
section 4 should state:

- MV is distinct from GMV in every window and from MSR in most;
- where MV and MSR coincide, it is at a single-asset corner;
- this is a symptom of the estimation-error sensitivity listed under *Shortcomings* in Table 3.2, not of
  a mis-specified objective.

---

## Pre-flight through the engine

Every strategy runs once through `run_backtest` on all 100 experiments. This checks that the signatures
plug into the loop and shows what the failure log (Table A.1) will contain. **Only failures are
inspected here.** Returns are discarded and reported in notebook 05.

In [12]:
runs = []
for name, fn in st.STRATEGIES.items():
    for i, e in experiments.iterrows():
        runs.append(bt.run_backtest(bt.experiment_prices(prices, e), fn, risk_free=panel.risk_free,
                                    experiment=i, strategy=name))
failures = bt.failure_log(runs)

print(f"Runs completed: {len(runs)} of {len(st.STRATEGIES) * len(experiments)}; none dropped.")
counts = failures.groupby("strategy").size().reindex(cfg.STRATEGY_ORDER, fill_value=0)
print("Optimisation failures by strategy:", counts.to_dict())
display(failures.assign(date=failures["date"].dt.date))

Runs completed: 800 of 800; none dropped.
Optimisation failures by strategy: {'EW': 0, 'GMV': 0, 'MV': 0, 'MSR': 7, 'IV': 0, 'ERC': 0, 'MDP': 0, 'MDC': 0}


,experiment,date,strategy,error,fallback
0,1,2022-09-30,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.00%/yr); tangency portfolio undefined,Held drifted weights (no trade)
1,1,2022-12-30,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.99%/yr); tangency portfolio undefined,Held drifted weights (no trade)
2,16,2019-01-07,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.93%/yr); tangency portfolio undefined,Held drifted weights (no trade)
3,49,2022-09-23,MSR,InfeasibleError: no asset has expected return above the risk-free rate (0.94%/yr); tangency portfolio undefined,Held drifted weights (no trade)
4,49,2022-12-22,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.90%/yr); tangency portfolio undefined,Held drifted weights (no trade)
5,88,2022-05-13,MSR,InfeasibleError: no asset has expected return above the risk-free rate (0.19%/yr); tangency portfolio undefined,Held drifted weights (no trade)
6,92,2018-11-19,MSR,InfeasibleError: no asset has expected return above the risk-free rate (1.81%/yr); tangency portfolio undefined,Held drifted weights (no trade)


The only failures are MSR windows in which no asset's trailing mean beat the T-bill rate. There, the
maximum-Sharpe portfolio does not exist, and the loop held the previous weights. These rows are Table
A.1 and appear in the appendix as they stand.

---

## Done-when checks

In [13]:
checks = []
checks.append(("D9 unit tests pass", (test_report["Result"] == "PASS").all()))
checks.append(("D9 eight functions, fixed order, interchangeable in the loop",
               list(st.STRATEGIES) == cfg.STRATEGY_ORDER and len(runs) == 8 * len(experiments)))
checks.append(("D10 Table 3.2 covers all eight, with a shortcomings column",
               list(spec.index) == cfg.STRATEGY_ORDER and "Shortcomings" in spec.columns))
checks.append(("D11 IV and ERC differ on the fixed window and the gap tracks correlation",
               (iv - erc).abs().max() > 1e-4 and slope_corr < -0.5))
checks.append(("D12 MV, GMV and MSR are three distinct vectors on the fixed window",
               min(distance(gmv, mv), distance(gmv, msr), distance(mv, msr)) > COINCIDE))
checks.append(("D12 every MV = MSR coincidence is a single-asset corner",
               bool((same["mv_single"] & same["msr_single"]).all())))
checks.append(("Only MSR fails, and only for want of an asset above r_f",
               set(failures["strategy"]) <= {"MSR"} and failures["error"].str.contains("risk-free").all()))

for label, ok in checks:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks), "workstream D exit criteria not met"

[PASS] D9 unit tests pass
[PASS] D9 eight functions, fixed order, interchangeable in the loop
[PASS] D10 Table 3.2 covers all eight, with a shortcomings column
[PASS] D11 IV and ERC differ on the fixed window and the gap tracks correlation
[PASS] D12 MV, GMV and MSR are three distinct vectors on the fixed window
[PASS] D12 every MV = MSR coincidence is a single-asset corner
[PASS] Only MSR fails, and only for want of an asset above r_f


---

## Handoff

Notebook 05 imports the strategies and runs the full grid:

```python
from src import backtest as bt, strategies as st

for name, fn in st.STRATEGIES.items():          # fixed strategy order
    ...  bt.run_backtest(block, fn, costs, risk_free=panel.risk_free, experiment=i, strategy=name)
```

| Item | Where it goes |
| --- | --- |
| Table 3.2 | Report section 4 |
| Tables 3.3–3.4 and the D11/D12 summaries | Section 4, one paragraph each; full tables in the appendix |
| MSR failures | Table A.1, via `bt.failure_log` in notebook 05 |
| λ = 3 kept; coincidences at corners | Section 4, stated explicitly |
| Ledoit–Wolf switch | Optional robustness row in workstream F |